# 01. Database Design & Relational Schema (Swiggy / Zomato Order System)

### Real-World Scenario
Designing an online food delivery platform schema with Customer details and Order transactions.

### Core Concepts
* **Primary Key (PK)**: `customer_id` and `order_id` uniquely identify records.
* **Foreign Key (FK)**: `orders.customer_id` references `customers.customer_id` to establish relational integrity.
* **Constraints**:
  * `NOT NULL`: Ensures crucial attributes (e.g., customer name, order total) cannot be left empty.
  * `CHECK`: Prevents invalid data entries (e.g., age must be $\ge 18$, order amount must be $> 0$).
  * `UNIQUE`: Enforces distinct phone numbers/emails across customers.

In [1]:
import sqlite3
import pandas as pd

# Initialize in-memory database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

# 1. Customers Table (Parent)
cursor.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    city TEXT NOT NULL,
    age INTEGER CHECK(age >= 18),
    phone TEXT UNIQUE NOT NULL
);
""")

# 2. Orders Table (Child)
cursor.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER,
    order_amount REAL CHECK(order_amount > 0),
    payment_mode TEXT CHECK(payment_mode IN ('UPI', 'Card', 'COD')),
    order_date DATE NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id) ON DELETE CASCADE
);
""")

# Populating Customers
customers_data = [
    ("Vetri", "Madurai", 22, "9876543210"),
    ("Sadhana", "Chennai", 21, "9876543211"),
    ("Arun", "Madurai", 24, "9876543212"),
    ("Kavya", "Coimbatore", 20, "9876543213"),
    ("Priya", "Chennai", 23, "9876543214"),
]
cursor.executemany(
    "INSERT INTO customers (name, city, age, phone) VALUES (?, ?, ?, ?)",
    customers_data,
)

# Populating Orders
orders_data = [
    (1, 450.0, "UPI", "2026-09-20"),
    (1, 250.0, "COD", "2026-09-21"),
    (2, 1200.0, "Card", "2026-09-22"),
    (3, 180.0, "UPI", "2026-09-22"),
    (2, 850.0, "UPI", "2026-09-24"),
    (4, 320.0, "COD", "2026-09-25"),
    (1, 600.0, "Card", "2026-09-26"),
]
cursor.executemany(
    "INSERT INTO orders (customer_id, order_amount, payment_mode, order_date) VALUES (?, ?, ?, ?)",
    orders_data,
)
conn.commit()

print("Real-World E-Commerce Schema Created & Populated!")

Real-World E-Commerce Schema Created & Populated!


###  Code Explanation
* **`PRAGMA foreign_keys = ON;`**: Enforces strict relational constraints in SQLite.
* **`CHECK(payment_mode IN ('UPI', 'Card', 'COD'))`**: Restricts field entries strictly to approved platform payment methods.
* **`ON DELETE CASCADE`**: Automatically removes associated order records if a customer deletes their profile.

# 02. Essential SQL Filtering, Grouping & Aggregations

### Real-World Query Execution
Filtering metrics to find cities generating high revenue with minimum average order value thresholds.

### Execution Order
1. `FROM` & `JOIN` (Identify tables)
2. `WHERE` (Filter individual orders before aggregation)
3. `GROUP BY` (Aggregate orders per city)
4. `HAVING` (Filter aggregated city stats)
5. `SELECT` (Compute metrics like sum/avg)
6. `ORDER BY` & `LIMIT` (Sort and slice top results)

In [2]:
def execute_sql(query: str) -> pd.DataFrame:
    return pd.read_sql_query(query, conn)


# Analyze Total Revenue and Average Spent per City for Orders paid via UPI/Card
query_city_analytics = """
SELECT 
    c.city,
    COUNT(o.order_id) AS total_orders,
    SUM(o.order_amount) AS total_revenue,
    ROUND(AVG(o.order_amount), 2) AS avg_order_value
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.payment_mode IN ('UPI', 'Card')
GROUP BY c.city
HAVING avg_order_value > 300
ORDER BY total_revenue DESC;
"""

print("--- City-wise Revenue Analytics ---")
print(execute_sql(query_city_analytics))

--- City-wise Revenue Analytics ---
      city  total_orders  total_revenue  avg_order_value
0  Chennai             2         2050.0           1025.0
1  Madurai             3         1230.0            410.0


###  Code Explanation
* **`WHERE o.payment_mode IN ('UPI', 'Card')`**: Filters out cash transactions prior to group operations.
* **`HAVING avg_order_value > 300`**: Filters groups post-aggregation to retain high-value customer locations.

# 03. Relational Joins (Customer Retention & Order Analysis)

### Real-World Application
Evaluating customer conversion by combining `customers` and `orders` data.

### Join Types
* **`INNER JOIN`**: Selects customers who have placed at least one order.
* **`LEFT JOIN`**: Retains all registered customers, identifying inactive users who haven't ordered yet (`order_id IS NULL`).
* **`CROSS JOIN`**: Generates all possible combinations for marketing campaigns or recommendation grids.

In [3]:
# 1. LEFT JOIN to identify inactive users
query_inactive_users = """
SELECT 
    c.customer_id,
    c.name,
    c.city,
    o.order_id
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;
"""

# 2. INNER JOIN for complete order histories
query_order_history = """
SELECT 
    o.order_id,
    c.name AS customer_name,
    o.order_amount,
    o.payment_mode,
    o.order_date
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
ORDER BY o.order_date ASC;
"""

print("--- Inactive Registered Customers ---")
print(execute_sql(query_inactive_users))

print("\n--- Complete Order Processing History ---")
print(execute_sql(query_order_history))

--- Inactive Registered Customers ---
   customer_id   name     city order_id
0            5  Priya  Chennai     None

--- Complete Order Processing History ---
   order_id customer_name  order_amount payment_mode  order_date
0         1         Vetri         450.0          UPI  2026-09-20
1         2         Vetri         250.0          COD  2026-09-21
2         3       Sadhana        1200.0         Card  2026-09-22
3         4          Arun         180.0          UPI  2026-09-22
4         5       Sadhana         850.0          UPI  2026-09-24
5         6         Kavya         320.0          COD  2026-09-25
6         7         Vetri         600.0         Card  2026-09-26


### Code Explanation
* **`LEFT JOIN ... WHERE o.order_id IS NULL`**: Standard anti-join pattern used to detect zero-order customers for promotional re-engagement.
* **`INNER JOIN`**: Excludes unengaged profiles, retrieving verified transactions.

# 04. Advanced SQL: CTEs, Subqueries, Views & Indexes

### Real-World Application
Building segmentation queries to identify VIP buyers (spending higher than platform average) and saving computed tables into reusable virtual Views.

In [5]:
# 1. CTE (Common Table Expression) + CASE Statement for Customer Categorization
query_vip_segmentation = """
WITH CustomerSpend AS (
    SELECT 
        c.customer_id,
        c.name,
        SUM(o.order_amount) AS total_spent
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id
)
SELECT 
    name,
    total_spent,
    CASE 
        WHEN total_spent >= 1000 THEN 'Platinum VIP'
        WHEN total_spent >= 500 THEN 'Gold Preferred'
        ELSE 'Regular'
    END AS customer_tier
FROM CustomerSpend
WHERE total_spent > (SELECT AVG(order_amount) FROM orders);
"""

# 2. Creating a dynamic database View for Reporting
cursor.execute("""
CREATE VIEW IF NOT EXISTS daily_sales_summary AS
SELECT 
    order_date,
    COUNT(order_id) AS total_orders,
    SUM(order_amount) AS daily_revenue
FROM orders
GROUP BY order_date;
""")

# 3. Index Creation for Performance Optimization
cursor.execute(
    "CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(customer_id);"
)

print("--- VIP Customer Segmentation via CTE ---")
print(execute_sql(query_vip_segmentation))

print("\n--- Sales Summary from View ---")
print(execute_sql("SELECT * FROM daily_sales_summary WHERE daily_revenue > 500;"))

--- VIP Customer Segmentation via CTE ---
      name  total_spent customer_tier
0    Vetri       1300.0  Platinum VIP
1  Sadhana       2050.0  Platinum VIP

--- Sales Summary from View ---
   order_date  total_orders  daily_revenue
0  2026-09-22             2         1380.0
1  2026-09-24             1          850.0
2  2026-09-26             1          600.0


###  Code Explanation
* **`WITH CustomerSpend AS (...)`**: Defines an intermediate named result set, making multi-stage aggregation readable.
* **`CREATE VIEW`**: Saves complex aggregation logic directly into the database engine for lightweight reporting calls.
* **`CREATE INDEX`**: Reduces query lookup complexity from $O(N)$ to $O(\log N)$ on high-volume foreign key filters.

# 05. SQL Window Functions (Leaderboards & Delta Tracking)

### Real-World Application
Calculating customer purchase order ranks within their respective cities and tracking consecutive order amount shifts using positional functions.

### Functions
* **`ROW_NUMBER()`**: Sequential ordering without ties.
* **`DENSE_RANK()`**: Ranks tied values equally without skipping subsequent integers.
* **`LAG()` / `LEAD()`**: Pulls metrics from previous/subsequent transactions to evaluate spending trends.

In [6]:
query_window_analytics = """
SELECT 
    c.city,
    c.name,
    o.order_amount,
    DENSE_RANK() OVER (PARTITION BY c.city ORDER BY o.order_amount DESC) AS city_rank,
    LAG(o.order_amount, 1) OVER (PARTITION BY c.customer_id ORDER BY o.order_date ASC) AS prev_order_amount,
    ROUND(o.order_amount - LAG(o.order_amount, 1) OVER (PARTITION BY c.customer_id ORDER BY o.order_date ASC), 2) AS spend_change
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id;
"""

print("--- City Ranking & Historic Order Delta ---")
print(execute_sql(query_window_analytics))

# Cleanup
conn.close()

--- City Ranking & Historic Order Delta ---
         city     name  order_amount  city_rank  prev_order_amount  \
0     Chennai  Sadhana        1200.0          1                NaN   
1     Chennai  Sadhana         850.0          2             1200.0   
2  Coimbatore    Kavya         320.0          1                NaN   
3     Madurai    Vetri         600.0          1              250.0   
4     Madurai    Vetri         450.0          2                NaN   
5     Madurai    Vetri         250.0          3              450.0   
6     Madurai     Arun         180.0          4                NaN   

   spend_change  
0           NaN  
1        -350.0  
2           NaN  
3         350.0  
4           NaN  
5        -200.0  
6           NaN  


###  Code Explanation
* **`DENSE_RANK() OVER (PARTITION BY c.city ...)`**: Evaluates order standings within individual cities independently.
* **`LAG(o.order_amount, 1)`**: Fetches the previous transaction value for each customer to calculate spending variance (`spend_change`).